# Train-Test Split

In this step, the dataset is divided into **training** and **test** subsets.

- The **training set** is used for:
  - exploratory data analysis (EDA),
  - feature engineering,
  - model training and validation.

- The **test set** is held out and used **only once at the end** to evaluate the final model performance on unseen data.

This separation ensures that the evaluation reflects how the model would perform in a real-world scenario, where future data is not available during model development.

---

## Why splitting early is important

To ensure a reliable and unbiased evaluation, the train-test split is performed **before any data-driven decisions are made**.

Many steps in the workflow — such as:
- feature selection,
- encoding strategies,
- transformation choices,
- and conclusions drawn from exploratory analysis,

can be influenced by patterns in the data.

If these decisions are based on the full dataset, information from the test set would indirectly influence the model. This leads to **data leakage**, resulting in overly optimistic performance estimates that do not generalize to new data.

By splitting the data early and conducting all analysis on the training set only, we ensure that:
- the test set remains a true proxy for unseen data,
- model evaluation is fair and realistic,
- and the overall workflow better reflects real-world deployment conditions.

---

## Practical note

Initial data inspection (such as checking structure, data types, and missing values) is performed on the full dataset to understand its composition.

However, from this point onward, all **data-driven decisions** are based exclusively on the training data.

In [1]:
import polars as pl

# Load data
df = pl.read_csv("./data/processed/01_do_telco_customer_churn_end.csv")

# Split by target
target_col = "Churn"  # adjust if needed

df_pos = df.filter(pl.col(target_col) == "Yes")
df_neg = df.filter(pl.col(target_col) == "No")

# Shuffle each group
df_pos = df_pos.sample(fraction=1.0, seed=42)
df_neg = df_neg.sample(fraction=1.0, seed=42)

# Split each group
split_pos = int(0.8 * df_pos.height)
split_neg = int(0.8 * df_neg.height)

train_df = pl.concat([
    df_pos[:split_pos],
    df_neg[:split_neg]
])

test_df = pl.concat([
    df_pos[split_pos:],
    df_neg[split_neg:]
])

# Shuffle again
train_df = train_df.sample(fraction=1.0, seed=42)
test_df = test_df.sample(fraction=1.0, seed=42)

In [2]:
def get_rate(df, target_col, positive_label="Yes"):
    counts = df.group_by(target_col).len()
    
    total = df.height
    positive = counts.filter(pl.col(target_col) == positive_label)["len"][0]
    
    return positive / total


train_rate = get_rate(train_df, "Churn")
test_rate = get_rate(test_df, "Churn")

print(f"Train churn rate: {train_rate:.4f}")
print(f"Test churn rate:  {test_rate:.4f}")
print(f"Difference:       {abs(train_rate - test_rate):.4f}")

Train churn rate: 0.2654
Test churn rate:  0.2654
Difference:       0.0001


In [3]:
train_df.write_csv("data/processed/02_train_dataset.csv")
test_df.write_csv("data/processed/02_test_dataset.csv")

## Conclusion

The dataset was successfully split into training and test subsets using a stratified approach, preserving the distribution of the target variable in both sets.

The training and test datasets have comparable sizes and very similar churn proportions, indicating that the split is representative of the overall population. This ensures that the model will be trained on data that reflects real-world patterns, while the test set remains a reliable proxy for unseen data.

From this point forward, all exploratory analysis, feature engineering, and model development will be performed exclusively on the training data. The test set will be used only for final evaluation, ensuring an unbiased assessment of model performance.